# Scenario comparison

Compares month-end **nominal** net worth across any number of scenarios against a
baseline, and finds each variant's break-even month — the point after which it
stays ahead for good.

Pick a baseline and up to 4 variant scenarios below — every chart and table in
this notebook updates automatically as the selection changes, no re-running
required. Each must already have a run (`analysis sim load <name> && analysis
sim run <name>`); the newest run for each is used.

> **Note:** after running cells, `git status` will show this notebook as modified
> even with no real changes — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it).



In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.connection import RunConnection
from analysis.sim.notebook_ui.pickers import dollar_plot_mode, multi_scenario_picker
from analysis.sim.notebook_ui.compare_plots import (
    comparability_widget, rules_and_events_widget,
    net_worth_compare_widget, breakeven_widget,
    year_end_widget,
)

MAX_VARIANTS = 4

# CONNECTIONS holds the live RunConnections, refreshed in place whenever either picker changes.
PICKER, CONNECTIONS = multi_scenario_picker(default_baseline="current", max_variants=MAX_VARIANTS)
PLOT_MODE = dollar_plot_mode()
PICKER

## Are these runs comparable?

A comparison is only meaningful if both runs came from the same code and the same
data version — otherwise a difference here could be an unrelated edit rather than
the scenario change under test.


In [ ]:
CONNECTIONS.show(comparability_widget)

## What actually differs?

Each scenario inherits everything from its parent except the rows it overrides, so
this is the whole mechanism behind any difference in the charts below. If a rule
you expected to change isn't listed, the override didn't take effect.

Two sources of difference are shown separately:

- **Rules** — as they stand at the *end* of the run. A calendar event that edits a
  rule mid-run (`set_amount`) shows up here as a changed amount, because the run
  snapshot records the post-event state.
- **Calendar events** — one-time interventions, including the benefit cut.


In [ ]:
CONNECTIONS.show(rules_and_events_widget)

In [ ]:
CONNECTIONS.show(rules_and_events_widget)

## Net worth over time

Use the shared **Dollar basis** toggle to switch between nominal and CPI-constant dollars. The selected basis applies to the net-worth chart and the comparison tables below.

In [ ]:
CONNECTIONS.show(net_worth_compare_widget, plot_mode=PLOT_MODE)

### Break-even

**Break-even** is the first month after which the variant stays ahead for the rest
of the scenario — not merely the first month it pokes above zero, which can happen
transiently when a lumpy payment lands in one scenario before the other.


In [ ]:
CONNECTIONS.show(breakeven_widget, plot_mode=PLOT_MODE)

## Account Comparisons

In [ ]:
from collections.abc import Sequence
from analysis.sim.connection import RunConnection
from analysis.sim.notebook_ui.pickers import MultiScenarioConnections
import ipywidgets as widgets
import pandas as pd


def compare_accounts(runs: MultiScenarioConnections) -> widgets.VBox:
    """
    Compare the accounts across multiple scenario connections.

    Parameters
    ----------
    runs : MultiScenarioConnections
        The multi-scenario connections object containing the scenarios to compare.

    Returns
    -------
    widgets.VBox
        The VBox widget containing the comparison display.
    """
    return widgets.VBox()

CONNECTIONS.show(compare_accounts)
v = [
        CONNECTIONS.connections[variant] for variant in CONNECTIONS.variants
    ]
base = CONNECTIONS.connections[CONNECTIONS.baseline]

def group_account_balances(base: RunConnection) -> dict[str, float]:
    """
    Get the account balances grouped by account.

    Parameters
    ----------
    base : RunConnection
        The run connection for the baseline scenario.

    Returns
    -------
    dict[str, float]
        A dictionary mapping account names to their monthly balances.
    """
    balances = base.items("account_balance").copy()
    balances['month'] = pd.to_datetime(balances['month']).dt.strftime("%Y-%m")

    return balances.pivot(index="month", columns="account", values="balance")


def compare_account_balance_histories(_, baseline_rc: RunConnection,
                                      variants_rcs: Sequence[RunConnection],
                                      **kwargs) -> None:
    ...

In [ ]:
import json
from typing import Any

from IPython.display import clear_output, Markdown

def compare_account_balances(_, baseline_rc: RunConnection,
                             variant_rcs: Sequence[RunConnection],
                             **kwargs) -> tuple[dict[str, set[str]], dict[str, set[str]], dict[str,Any], dict[str, Any]]:
    """
    Compare the account balances across multiple scenario connections.

    Parameters
    ----------
    _ : Any
        Placeholder for the first argument.
    baseline_rc : RunConnection
        The run connection for the baseline scenario.
    variants_rcs : Sequence[RunConnection]
        The run connections for the variant scenarios.
    **kwargs : dict
        Additional keyword arguments.

    Returns
    -------
    dict[str, set[str]], dict[str, set[str]], dict[str, dict[str, ...]], dict[str, ...]
    """
    bg = group_account_balances(baseline_rc)
    vgs = {
        name: group_account_balances(variant)
        for name, variant in variant_rcs.items()
        if variant is not baseline_rc}
    bg_keys = set(bg.keys())
    vgs_keys = {
        variant: set(groups.keys())
        for variant, groups in vgs.items()
    }
    added = {variant: keys - bg_keys for variant, keys in vgs_keys.items()}
    deleted = {variant: bg_keys - keys for variant, keys in vgs_keys.items()}
    diff = {
        variant: {
            acct: (bg[acct], vgs[variant][acct])
            for acct in bg_keys
            if (bg[acct] != vgs[variant][acct]).any()
        }
        for variant in vgs_keys.keys()
        for acct in bg_keys
        if (bg[acct] != vgs[variant][acct]).any()
    }
    return (added, deleted, diff, bg)


def to_markdown(*lines: str) -> widgets.Output:
    out = widgets.Output()
    with out:
        display(
            Markdown("\n".join(lines))
        )
    return out


def account_balance_diffs(baseline: str,
                          baseline_rc: RunConnection,
                          variant_rcs: Sequence[RunConnection],
                          **kwargs) -> None:
    added, deleted, diff, bg = compare_account_balances(None, baseline_rc, variant_rcs)
    diff_accounts = {
        key
        for d in diff.values()
        for key in d.keys()
    }
    diff_months = {
        m
        for d in diff.values()
        for key, (base, variant) in d.items()
        for m in base.index
        if base[m] != variant[m]
    }
    diff_accounts = set(sorted(diff_accounts))
    diff_months = list(sorted(diff_months))

    accounts = baseline_rc.items("account")
    accounts = accounts.set_index(accounts["name"], drop=True)

    metas = accounts["meta_json"].apply(json.loads)
    builtins = {
        acct for acct in metas.keys()
        if metas[acct].get("builtin", False)
    }
    transients = {
        acct for acct in metas.keys()
        if metas[acct].get("diff", None) == "transient"
    }
    balance_sheet_accts = {
        acct
        for acct in accounts.index
        if accounts.loc[acct, "kind"] in ("asset", "liability")
    }
    excluded = builtins | balance_sheet_accts | transients

    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        d_box = to_markdown(f"## Comparing `{baseline}` with `{", ".join(variant_rcs.keys())}`")
        d_add = (
            to_markdown(f"-- Added accounts in `{variant}`: {", ".join(added[variant] - excluded)}")
            for variant in variant_rcs.keys()
        )
        d_del= (
            to_markdown(f"-- Deleted accounts in `{variant}`: {", ".join(deleted[variant] - excluded)}")
            for variant in variant_rcs.keys()
        )
        def diff_acct(acct: str):
            base_col = bg[acct]
            def diffs_(variant: str):
                vaccts = diff.get(variant, None)
                if vaccts is None:
                    return ()
                return tuple(
                    c
                    for cs in (
                        (
                            col.copy().to_frame(variant),
                            delta.to_frame(f"Δ{variant}")
                        )
                        for col in vaccts.get(acct, ())
                        for delta in ((col - base_col),)
                        if delta.any()
                    )
                    if len(cs) > 0
                    for c in cs
                )
            dd = [diffs_(variant) for variant in variant_rcs.keys()]
            if len(dd) == 0:
                return ()
            retval = (
                base_col.to_frame("baseline"),
                *(
                    x
                    for d in dd
                    for x in d
                )
            )
            return retval
        return {
            acct: cl[0].join(cl[1:])
            for acct in (diff_accounts - excluded)
            for cl in (diff_acct(acct),)
            if len(cl) > 1
        }

def _show_account_balance_diffs(diffs: dict[str, pd.DataFrame]) -> widgets.VBox:
    def show(label: str, df: pd.DataFrame) -> widgets.Output:
        df = df.loc[(df != 0).all(axis=1)]
        df = df / 100.0
        df = df.style.format("${:.2f}")
        out = widgets.Output()
        with out:
            display(df)
        return (to_markdown(f"### {label}"), out)
    outs = [
        b
        for acct, cl in diffs.items()
        for b in show(acct, cl)
    ]
    return widgets.VBox(outs)

def show_account_balance_diffs(_,
                          baseline: str,
                          baseline_rc: RunConnection,
                          variant_rcs: Sequence[RunConnection],
                          **kwargs) -> pd.DataFrame:
    diffs = account_balance_diffs(
        baseline=baseline,
        baseline_rc=baseline_rc,
        variant_rcs=variant_rcs,
        **kwargs
    )
    return _show_account_balance_diffs(diffs)

CONNECTIONS.show(show_account_balance_diffs)

### Annual net worth comparisons and deltas.|

In [ ]:
CONNECTIONS.show(year_end_widget, plot_mode=PLOT_MODE)